In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kendalltau
from itertools import combinations
import re
import matplotlib.pyplot as plt

from statsmodels.stats.multitest import multipletests
import cobra

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Calibri'] + plt.rcParams['font.sans-serif']

LONG_CSV = '/path/to/avg_ig_by_region.csv'
LOOKUP_CSV = '/path/to/fs_lookup.csv'
LOBE_CSV = '/path/to/destrieux - lobe.csv'

VALUE_COL = 'avg_saliency'
THRESHOLD = 0.6        
N_CLUSTERS = 8
N_LABELS = 30
N_ITERATIONS = 200

EXCLUDE = {'medial_wall', 'unknown'}   
NEW_LOBE_ORDER = ['frontal', 'temporal', 'parietal', 'occipital', 'limbic']

LOBE_RGBA = {
    "frontal":   (234/255,  67/255,  53/255, 1.0),
    "parietal":  (227/255, 116/255,   0/255, 1.0),
    "occipital": ( 66/255, 103/255, 210/255, 1.0),
    "temporal":  ( 52/255, 168/255,  83/255, 1.0),
    "limbic":    (255/255, 194/255,   0/255, 1.0),
    "sub":       (150/255,  85/255, 132/255, 1.0),
}
UNKNOWN_RGBA = (0.72, 0.72, 0.72, 1.0)

long_df = pd.read_csv(LONG_CSV)
lookup = pd.read_csv(LOOKUP_CSV, usecols=['code', 'region'])
lobe_df = pd.read_csv(LOBE_CSV)

area = list(lobe_df['area'])
labels_ref = ['lh_' + a for a in area] + ['rh_' + a for a in area][::-1]
area_to_lobe = dict(zip(lobe_df['area'], lobe_df['lobe']))


def strip_hemisphere(region_name):
    return re.sub(r'^(lh|rh)_', '', region_name)


def region_lobe(region_name):
    return area_to_lobe.get(strip_hemisphere(region_name))


def region_color(region_name):
    return LOBE_RGBA.get(str(region_lobe(region_name)), UNKNOWN_RGBA)


def is_excluded(region_name):
    return strip_hemisphere(region_name).lower() in EXCLUDE


data = long_df.pivot_table(index='subject', columns='region_id',
                           values=VALUE_COL, aggfunc='first', dropna=False)

code_to_region = dict(zip(lookup['code'], lookup['region']))
data.columns = [re.sub(r'^ctx_', '', code_to_region.get(c, str(c))) for c in data.columns]
data = data.apply(pd.to_numeric, errors='coerce')

available = [l for l in labels_ref if l in data.columns and not is_excluded(l)]
missing = [l for l in labels_ref if l not in data.columns]
dropped = [l for l in labels_ref if l in data.columns and is_excluded(l)]
if dropped:
    print(f"Excluded {len(dropped)} regions:", dropped)
if missing:
    print(f"{len(missing)} regions in labels_ref not found in data — dropped:", missing)

data = data[available]

lh_labels = [l for l in available if l.startswith('lh_')]
rh_labels = [l for l in available if l.startswith('rh_')]
lh_sorted = [l for lobe in NEW_LOBE_ORDER for l in lh_labels if region_lobe(l) == lobe]
rh_sorted = [l for lobe in reversed(NEW_LOBE_ORDER) for l in rh_labels if region_lobe(l) == lobe]
labels = lh_sorted + rh_sorted

lost = set(available) - set(labels)
if lost:
    print(f"WARNING: {len(lost)} regions dropped by the lobe sort "
          f"(their lobe is not in NEW_LOBE_ORDER): {sorted(lost)}")

data = data[labels]
n = len(labels)


def build_connectivity(data, labels, alpha=0.05, method='fdr_bh',
                       positive_only=True, min_n=3):
    n = len(labels)
    pairs, taus, pvals = [], [], []

    for i, j in combinations(range(n), 2):
        xi, yj = labels[i], labels[j]
        pair = data[[xi, yj]].dropna()
        if len(pair) < min_n:
            continue
        tau, p = kendalltau(pair[xi], pair[yj])
        if np.isnan(p):
            continue
        pairs.append((i, j))
        taus.append(tau)
        pvals.append(p)

    taus = np.asarray(taus)
    pvals = np.asarray(pvals)

    reject, qvals, _, _ = multipletests(pvals, alpha=alpha, method=method)
    keep = reject & (taus > 0) if positive_only else reject

    conn = np.zeros((n, n))
    for (i, j), t, k in zip(pairs, taus, keep):
        if k:
            conn[i, j] = conn[j, i] = abs(t)

    print(f"{keep.sum()} / {len(pvals)} pairs survive {method} at q<{alpha}")
    return conn, pairs, taus, pvals, qvals


connectivity_array, pairs, taus, pvals, qvals = build_connectivity(data, labels)

n_edges = int((connectivity_array > THRESHOLD).sum() // 2)
print(f"{n_edges} edges above THRESHOLD={THRESHOLD} will be drawn")

cluster_labels, reordered_matrix, reordered_regions = cobra.cluster.create_main_clustering_visualization(
    connectivity_array,
    labels,
    n_clusters=N_CLUSTERS,
)

colors = [region_color(l) for l in labels]

G, pos = cobra.network.make_network_graph(
    connectivity_array, labels, cluster_labels,
    threshold=THRESHOLD,
    node_colors=colors,
    color_by='custom',
    show_labels='top_n',
    orientation='horizontal',
    top_n_labels=N_LABELS,
    n_interations=N_ITERATIONS,
)